In [63]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [2]:
cwd = Path.cwd()
project_root = cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print("Done!")

In [3]:
from credit_risk.dataset import AFTER_EDA, load_splits

2026-06-05 08:46:24.087 | INFO     | credit_risk.config:<module>:11 - PROJ_ROOT path is: /Users/ak007/SML/Credit-Risk-Default-Prediction-System


In [4]:
train_df, val_df, test_df, metadata = load_splits(path=AFTER_EDA)

2026-06-05 08:46:24.406 | INFO     | credit_risk.dataset:load_splits:263 - Checking if the files exists...
2026-06-05 08:46:24.417 | INFO     | credit_risk.dataset:load_splits:265 - Loading the Cached files...
2026-06-05 08:46:24.743 | INFO     | credit_risk.dataset:load_splits:273 - Loaded sucessfully all the splits and the metadata, Train_df shape: (466071, 110), val_df shape: (420204, 110), test_df shape: (431712, 110)


In [5]:
train_df.shape, val_df.shape, test_df.shape

((466071, 110), (420204, 110), (431712, 110))

In [6]:
# let's first check the null percentages in the columns
train_null = train_df.isna().mean(axis=0).sort_values(ascending=False)*100

In [7]:
train_null.head(40)

inq_fi                                 100.000000
sec_app_fico_range_low                 100.000000
annual_inc_joint                       100.000000
dti_joint                              100.000000
verification_status_joint              100.000000
open_acc_6m                            100.000000
open_il_12m                            100.000000
open_il_24m                            100.000000
mths_since_rcnt_il                     100.000000
total_bal_il                           100.000000
il_util                                100.000000
open_rv_12m                            100.000000
open_rv_24m                            100.000000
max_bal_bc                             100.000000
all_util                               100.000000
total_cu_tl                            100.000000
inq_last_12m                           100.000000
revol_bal_joint                        100.000000
open_act_il                            100.000000
sec_app_fico_range_high                100.000000


In [8]:
all_null = list(train_null[train_null == 100.0].index)

In [9]:
len(all_null)

30

In [10]:
all_null

['inq_fi',
 'sec_app_fico_range_low',
 'annual_inc_joint',
 'dti_joint',
 'verification_status_joint',
 'open_acc_6m',
 'open_il_12m',
 'open_il_24m',
 'mths_since_rcnt_il',
 'total_bal_il',
 'il_util',
 'open_rv_12m',
 'open_rv_24m',
 'max_bal_bc',
 'all_util',
 'total_cu_tl',
 'inq_last_12m',
 'revol_bal_joint',
 'open_act_il',
 'sec_app_fico_range_high',
 'sec_app_inq_last_6mths',
 'sec_app_mths_since_last_major_derog',
 'sec_app_chargeoff_within_12_mths',
 'sec_app_num_rev_accts',
 'sec_app_open_act_il',
 'sec_app_revol_util',
 'sec_app_open_acc',
 'sec_app_mort_acc',
 'sec_app_collections_12_mths_ex_med',
 'sec_app_earliest_cr_line']

In [11]:
train_df.drop(columns=all_null, inplace=True)
val_df.drop(columns=all_null, inplace=True)
test_df.drop(columns=all_null, inplace=True)

In [12]:
y_train = train_df['target']
y_val = val_df['target']
y_test = test_df['target']

In [13]:
X_train = train_df.drop(columns=["target"])
X_val = val_df.drop(columns=["target"])
X_test = test_df.drop(columns=["target"])

In [14]:
assert (X_train.shape[1] == X_val.shape[1]) and (X_val.shape[1] == X_test.shape[1])

In [15]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 466071 entries, 1117060 to 1939378
Data columns (total 79 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   loan_amnt                       466071 non-null  float64       
 1   funded_amnt                     466071 non-null  float64       
 2   funded_amnt_inv                 466071 non-null  float64       
 3   term                            466071 non-null  str           
 4   int_rate                        466071 non-null  float64       
 5   installment                     466071 non-null  float64       
 6   grade                           466071 non-null  str           
 7   sub_grade                       466071 non-null  str           
 8   emp_title                       438488 non-null  str           
 9   emp_length                      445069 non-null  str           
 10  home_ownership                  466071 non-null  str           
 

In [16]:
X_train.describe()

,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,issue_d,dti,delinq_2yrs,fico_range_low,...,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit
count,466071.000000,466071.000000,466071.00000,466071.000000,466071.000000,4.660670e+05,466071,466071.000000,466042.000000,466071.000000,...,395795.000000,395795.000000,395642.000000,411955.000000,464706.000000,465966.000000,3.957950e+05,4.160410e+05,4.160410e+05,3.957950e+05
mean,14313.301461,14287.813230,14218.32476,13.826272,431.987386,7.326826e+04,2013-08-23 11:04:39.946188,17.217138,0.284599,696.118510,...,0.082108,1.918150,94.709580,51.974091,0.106663,0.033844,1.683230e+05,4.598811e+04,2.012186e+04,3.764764e+04
min,500.000000,500.000000,0.00000,5.420000,4.930000,1.896000e+03,2007-06-01 00:00:00,0.000000,0.000000,610.000000,...,0.000000,0.000000,15.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8000.000000,8000.000000,8000.00000,10.990000,256.610000,4.500000e+04,2013-03-01 00:00:00,11.360000,0.000000,675.000000,...,0.000000,1.000000,92.000000,25.000000,0.000000,0.000000,4.652800e+04,1.989000e+04,7.400000e+03,1.205350e+04
50%,12000.000000,12000.000000,12000.00000,13.660000,379.760000,6.300000e+04,2014-01-01 00:00:00,16.870000,0.000000,690.000000,...,0.000000,2.000000,100.000000,50.000000,0.000000,0.000000,1.103890e+05,3.497900e+04,1.420000e+04,2.824900e+04
75%,20000.000000,20000.000000,19950.00000,16.490000,566.500000,8.886150e+04,2014-07-01 00:00:00,22.780000,0.000000,710.000000,...,0.000000,3.000000,100.000000,80.000000,0.000000,0.000000,2.459310e+05,5.770300e+04,2.630000e+04,5.091350e+04
max,35000.000000,35000.000000,35000.00000,26.060000,1409.990000,7.500000e+06,2014-12-01 00:00:00,39.990000,29.000000,845.000000,...,24.000000,26.000000,100.000000,100.000000,12.000000,63.000000,9.999999e+06,2.688920e+06,1.090700e+06,1.241783e+06
std,8285.328704,8273.171262,8296.41226,4.357272,243.491216,5.496616e+04,NaN,7.850806,0.797234,30.773018,...,0.445473,1.573425,8.066187,34.611561,0.332642,0.329736,1.701041e+05,4.362475e+04,1.964710e+04,3.973245e+04


In [17]:
# there is one more columns that should be converted in datetime
X_train['earliest_cr_line'] = pd.to_datetime(X_train['earliest_cr_line'], format="%b-%Y")

In [18]:
# we will drop policy_code cause it doesn't have any signal/variance that model can learn from
X_train.drop(columns='policy_code', inplace=True)

In [19]:
numeric_cols = X_train.select_dtypes(include='number').columns
numeric_cols

Index(['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'int_rate',
       'installment', 'annual_inc', 'dti', 'delinq_2yrs', 'fico_range_low',
       'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq',
       'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal',
       'revol_util', 'total_acc', 'collections_12_mths_ex_med',
       'mths_since_last_major_derog', 'acc_now_delinq', 'tot_coll_amt',
       'tot_cur_bal', 'total_rev_hi_lim', 'acc_open_past_24mths',
       'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths',
       'delinq_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op',
       'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc',
       'mths_since_recent_bc', 'mths_since_recent_bc_dlq',
       'mths_since_recent_inq', 'mths_since_recent_revol_delinq',
       'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl',
       'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl',
       'num_rev_accts', 'num_rev_tl_bal_gt_0', 'n

In [20]:
string_cols = X_train.select_dtypes(include='str').columns
string_cols

Index(['term', 'grade', 'sub_grade', 'emp_title', 'emp_length',
       'home_ownership', 'verification_status', 'desc', 'purpose', 'title',
       'zip_code', 'addr_state', 'initial_list_status', 'application_type',
       'disbursement_method'],
      dtype='str')

In [21]:
datetime_cols = X_train.select_dtypes(include=['datetime']).columns
datetime_cols

Index(['issue_d', 'earliest_cr_line'], dtype='str')

In [22]:
assert len(numeric_cols) + len(string_cols) + len(datetime_cols) == len(X_train.columns)

In [23]:
string_cols

Index(['term', 'grade', 'sub_grade', 'emp_title', 'emp_length',
       'home_ownership', 'verification_status', 'desc', 'purpose', 'title',
       'zip_code', 'addr_state', 'initial_list_status', 'application_type',
       'disbursement_method'],
      dtype='str')

In [24]:
# creating some derived features
# credit age years = issue_d - earliest_cr_line
train_null[["issue_d", 'earliest_cr_line']]

issue_d             0.000000
earliest_cr_line    0.006222
dtype: float64

In [25]:
credit_age = (X_train['issue_d'] - X_train['earliest_cr_line']).dt.days / 365.25
credit_age

1117060    25.248460
1117061    20.334018
1117062    22.080767
1117063    12.334018
1117064    14.165640
             ...    
1939374    23.249829
1939375     8.000000
1939376    11.331964
1939377    10.417522
1939378     8.914442
Length: 466071, dtype: float64

In [26]:
X_train['credit_age_yrs'] = credit_age

In [27]:
# fico_mid = average of fico_low and fico_high
X_train[['fico_range_low', 'fico_range_high']].notna().all()

fico_range_low     True
fico_range_high    True
dtype: bool

In [28]:
fico_mid = (X_train['fico_range_low'] + X_train['fico_range_high'])/2
fico_mid

1117060    712.0
1117061    752.0
1117062    682.0
1117063    687.0
1117064    667.0
           ...  
1939374    702.0
1939375    682.0
1939376    722.0
1939377    677.0
1939378    717.0
Length: 466071, dtype: float64

In [29]:
X_train['fico_mid'] = fico_mid

In [39]:
X_train.drop(columns=['issue_d', 'earliest_cr_line', 'fico_range_low', 'fico_range_high'], inplace=True)

### Let's first build a baseline model, A very simple default model, then we will iterate

#### Numeric columns

In [41]:
train_null = ((X_train.select_dtypes(include='number').isna().sum(axis=0)/len(X_train))*100).sort_values(ascending=False)

In [42]:
train_null[train_null > 0]

mths_since_last_record            86.567068
mths_since_last_major_derog       78.774049
mths_since_recent_bc_dlq          78.766969
mths_since_recent_revol_delinq    70.151543
mths_since_last_delinq            53.693107
mths_since_recent_inq             19.755145
mo_sin_old_il_acct                17.931817
num_tl_120dpd_2m                  16.819111
pct_tl_nvr_dlq                    15.111217
avg_cur_bal                       15.080964
mo_sin_old_rev_tl_op              15.078604
mo_sin_rcnt_rev_tl_op             15.078604
num_accts_ever_120_pd             15.078389
tot_coll_amt                      15.078389
num_actv_bc_tl                    15.078389
num_actv_rev_tl                   15.078389
total_rev_hi_lim                  15.078389
tot_cur_bal                       15.078389
num_bc_tl                         15.078389
num_il_tl                         15.078389
num_op_rev_tl                     15.078389
tot_hi_cred_lim                   15.078389
num_rev_accts                   

#### categorical columns

In [56]:
X_train.drop(columns=['application_type', 'disbursement_method'], inplace=True)

In [58]:
X_train.select_dtypes(include='str').columns

Index(['term', 'grade', 'sub_grade', 'emp_title', 'emp_length',
       'home_ownership', 'verification_status', 'desc', 'purpose', 'title',
       'zip_code', 'addr_state', 'initial_list_status'],
      dtype='str')

In [60]:
X_train['purpose'].nunique()

14

In [57]:
assert len(X_train.select_dtypes(include='number').columns) + len(X_train.select_dtypes(include='str').columns) == len(X_train.columns)

In [65]:
NUMERICAL_COLS = X_train.select_dtypes(include='number').columns.to_list()
CATEGORICAL_COLS = ['term', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status']

In [69]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, NUMERICAL_COLS),
    ('cat', categorical_pipeline, CATEGORICAL_COLS),
], remainder='drop')

In [70]:
preprocessor.fit(X_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

Transformation -> feature_engineering -> drop_unusable_features -> fit -> transform -> modelling

In [72]:
# we will have to build the extra features, then drop the columns fo the val and test set too.
'policy_code' in X_val.columns and 'policy_code' in X_test.columns

True

In [73]:
# let's first create the features
X_val['earliest_cr_line'] = pd.to_datetime(X_val['earliest_cr_line'], format='%b-%Y')
X_test['earliest_cr_line'] = pd.to_datetime(X_test['earliest_cr_line'], format='%b-%Y')

In [74]:
# feature engineering (credit years and fico_mid)
X_val['credit_age_yrs'] = (X_val['issue_d'] - X_val['earliest_cr_line']).dt.days/365.25
X_test['credit_age_yrs'] = (X_test['issue_d'] - X_test['earliest_cr_line']).dt.days/365.25

X_val['fico_mid'] = (X_val['fico_range_low'] + X_val['fico_range_high'])/2
X_test['fico_mid'] = (X_test['fico_range_low'] + X_test['fico_range_high'])/2

In [79]:
all_null

['inq_fi',
 'sec_app_fico_range_low',
 'annual_inc_joint',
 'dti_joint',
 'verification_status_joint',
 'open_acc_6m',
 'open_il_12m',
 'open_il_24m',
 'mths_since_rcnt_il',
 'total_bal_il',
 'il_util',
 'open_rv_12m',
 'open_rv_24m',
 'max_bal_bc',
 'all_util',
 'total_cu_tl',
 'inq_last_12m',
 'revol_bal_joint',
 'open_act_il',
 'sec_app_fico_range_high',
 'sec_app_inq_last_6mths',
 'sec_app_mths_since_last_major_derog',
 'sec_app_chargeoff_within_12_mths',
 'sec_app_num_rev_accts',
 'sec_app_open_act_il',
 'sec_app_revol_util',
 'sec_app_open_acc',
 'sec_app_mort_acc',
 'sec_app_collections_12_mths_ex_med',
 'sec_app_earliest_cr_line']

ok, I think we don't really need to drop any columns from the val or test after creating the features that we did with the train, cause the preprocessor will itself drop the extra features.

In [80]:
X_train_feat = preprocessor.transform(X_train)
X_val_feat = preprocessor.transform(X_val)
X_test_feat = preprocessor.transform(X_test)

In [82]:
X_train_feat.shape, X_val_feat.shape, X_test_feat.shape

((466071, 192), (420204, 192), (431712, 192))

In [84]:
print(np.isnan(X_train_feat).sum())
print(np.isnan(X_val_feat).sum())
print(np.isnan(X_test_feat).sum())

0
0
0


In [86]:
X_train_feat.dtype, X_val_feat.dtype, X_test_feat.dtype

(dtype('float64'), dtype('float64'), dtype('float64'))